# ReefScan-ID: Direct 5-class SegFormer-B0 (E4 a)

SegFormer-B0 trained to label every pixel as coral healthy, coral bleached, coral dead, substrate or other, for the direct side of E4. The data, crops, loss and scoring follow notebook 02, with the 5-class mapping of the 2026-11-19 decision. The test split is not used. Run in Google Colab with a GPU runtime.

### Setup: Colab runtime and Google Drive

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU: {}".format(gpu_name))
print("CPU count: {}".format(os.cpu_count()))

base_dir = "/content/drive/MyDrive/ReefScan-ID"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))

### Setup: libraries

In [ ]:
!pip install -q datasets
import random
import time
import numpy as np
from PIL import Image

import datasets
print("torch version: {}".format(torch.__version__))
print("datasets version: {}".format(datasets.__version__))
print("numpy version: {}".format(np.__version__))

### Settings

The optimiser settings are copied from notebook 02: AdamW, learning rate 6e-5, weight decay 0.01, linear decay to 0. The number of epochs is set after the timing cell.

In [ ]:
crop_size = 512
batch_size = 8
num_workers = 2
val_height = 512
val_width = 1024
val_batch_size = 4
seed = 0

learning_rate = 6e-5
weight_decay = 0.01
timing_steps = 30

# Reference values printed by notebook 02
train_pixels_reference = 3181379584
segformer3_params = 3714915

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

print("Crops: {} by {}, batch size {}, workers {}".format(crop_size, crop_size, batch_size, num_workers))
print("Validation photos resized to {} by {}, batch size {}".format(val_height, val_width, val_batch_size))
print("Optimiser: AdamW, learning rate {}, weight decay {}, linear decay to 0 (as in notebook 02)".format(
    learning_rate, weight_decay))
print("The number of epochs is not set here")
print("The test split is not used in this notebook")

## Section 1: Data, 5-class mapping, model and timing

Question: does the direct model see the same data, crops, loss and scoring as the 3-class SegFormer-B0 of notebook 02 with the 5-class mapping of the 2026-11-19 decision, how are the train pixels spread over the five classes, and how long does one epoch take on this GPU?

Checks: load Coralscapes from the Drive cache, build the 5-class mapping and compare it with notebook 02's coral group, count the train pixels and compute the class weights with the formula of notebook 02, build the crop and validation loaders, build SegFormer-B0 with five classes and run one forward pass, score the untrained model once, then time 30 training steps and one validation pass to estimate the seconds per epoch. Only the train and validation splits are used.

### Load the dataset

In [ ]:
raw_dir = base_dir + "/RawData"

from datasets import load_dataset

print("Cache folder exists: {}".format(os.path.isdir(raw_dir)))

start = time.time()
ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=raw_dir)
print(ds)
print("Loading took {:.1f} seconds".format(time.time() - start))

### Class names from the dataset card

Ids and names copied from notebook 02, which copied them from the table on the Coralscapes dataset card.

In [ ]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

### Class groups

The coral ids are split into healthy, bleached and dead with the stage 2 state map of notebook 04. Substrate and the coral group come from the stage 1 map of notebook 02. The excluded coral ids come from `settings.json`.

In [ ]:
import json

# Stage 2 state map of notebook 04
healthy_ids = [6, 17, 22, 25, 28, 31, 34, 36]
bleached_ids = [4, 16, 19, 33]
dead_ids = [3, 20, 23, 32, 37]

# Stage 1 groups of notebook 02
substrate_ids = [5, 10, 12, 18]
coral_ids = [3, 4, 6, 16, 17, 19, 20, 21, 22, 23, 25, 27, 28, 31, 32, 33, 34, 36, 37]

with open(base_dir + "/Processed/stage2_patches/settings.json") as handle:
    settings = json.load(handle)
excluded_ids = settings["excluded_ids"]

print("Healthy ids: {}".format(healthy_ids))
print("Bleached ids: {}".format(bleached_ids))
print("Dead ids: {}".format(dead_ids))
print("Substrate ids: {}".format(substrate_ids))
print("Coral ids of notebook 02: {}".format(coral_ids))
print("Excluded ids from settings.json: {}".format(excluded_ids))

### Build the 5-class mapping

In [ ]:
group_names = {0: "coral healthy", 1: "coral bleached", 2: "coral dead", 3: "substrate", 4: "other"}
ignore_id = 255

# Everything on the card starts as other. Value 0 and the excluded coral ids are ignored
mapping = np.full(256, ignore_id, dtype=np.uint8)
for value in class_names:
    mapping[value] = 4
mapping[substrate_ids] = 3
mapping[healthy_ids] = 0
mapping[bleached_ids] = 1
mapping[dead_ids] = 2
mapping[excluded_ids] = ignore_id
mapping[0] = ignore_id

for group_id, group_name in group_names.items():
    names = [class_names[v] for v in class_names if mapping[v] == group_id]
    print("{} {} ({} ids): {}".format(group_id, group_name, len(names), names))
ignored = [class_names[v] for v in class_names if mapping[v] == ignore_id]
print("ignored ({} ids on the card): {}".format(len(ignored), ignored))
print("Mask value 0 maps to: {}".format(mapping[0]))

coral_five = set(healthy_ids + bleached_ids + dead_ids)
print("No id is in two classes: {}".format(
    len(healthy_ids + bleached_ids + dead_ids + substrate_ids) == len(coral_five | set(substrate_ids))))
print("Coral classes plus the ignored coral ids equal notebook 02's coral group: {}".format(
    coral_five | set(excluded_ids) == set(coral_ids)))
print("The excluded ids of settings.json are 21 and 27, as in the decision: {}".format(sorted(excluded_ids) == [21, 27]))
print("Class 4 holds the ids that are neither coral nor substrate in notebook 02: {}".format(
    {v for v in class_names if mapping[v] == 4} == set(class_names) - set(coral_ids) - set(substrate_ids)))

### Count train pixels per mask value

In [ ]:
pixel_counts = {}

start = time.time()
counts = np.zeros(256, dtype=np.int64)
masks = ds["train"].select_columns(["label"])
for row in masks:
    mask = np.array(row["label"])
    counts += np.bincount(mask.ravel(), minlength=256)
pixel_counts["train"] = counts

print("train: {} masks counted, {} pixels, {:.0f} seconds".format(masks.num_rows, counts.sum(), time.time() - start))
print("Same pixel total as notebook 02: {}".format(int(counts.sum()) == train_pixels_reference))

### Pixel share per class

In [ ]:
for split in ["train"]:
    counts = pixel_counts[split]
    total = counts.sum()
    for group_id, group_name in group_names.items():
        share = 100 * counts[mapping == group_id].sum() / total
        print("{} {}: {:.2f}% of all pixels".format(split, group_name, share))
    ignored = 100 * counts[mapping == ignore_id].sum() / total
    print("{} ignored: {:.2f}% of all pixels".format(split, ignored))

### Class weights and loss

In [ ]:
train_counts = pixel_counts["train"]
group_pixels = np.array([train_counts[mapping == group_id].sum() for group_id in group_names])
group_share = group_pixels / group_pixels.sum()
class_weights = 1 / group_share
class_weights = class_weights / class_weights.mean()

for group_id, group_name in group_names.items():
    print("{}: {:.2f}% of labelled train pixels, weight {:.3f}".format(
        group_name, 100 * group_share[group_id], class_weights[group_id]))

weight_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)


def segmentation_loss(logits, masks):
    upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
    return torch.nn.functional.cross_entropy(upsampled, masks, weight=weight_tensor, ignore_index=ignore_id)

### Crop dataset class

In [ ]:
import random
from torch.utils.data import Dataset

crop_size = 512


class CropDataset(Dataset):
    def __init__(self, split, crop_size, flip):
        self.split = split
        self.crop_size = crop_size
        self.flip = flip

    def __len__(self):
        return self.split.num_rows

    def __getitem__(self, index):
        row = self.split[index]
        image = np.array(row["image"])
        mask = mapping[np.array(row["label"])]

        # Same random window for the image and the mask
        top = random.randint(0, image.shape[0] - self.crop_size)
        left = random.randint(0, image.shape[1] - self.crop_size)
        image = image[top:top + self.crop_size, left:left + self.crop_size]
        mask = mask[top:top + self.crop_size, left:left + self.crop_size]

        if self.flip and random.random() < 0.5:
            image = image[:, ::-1]
            mask = mask[:, ::-1]

        image = torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255
        mask = torch.from_numpy(mask.copy()).long()
        return image, mask


print("Crop size: {}".format(crop_size))

### Validation dataset class

In [ ]:
class ResizeDataset(Dataset):
    def __init__(self, split, height, width):
        self.split = split
        self.height = height
        self.width = width

    def __len__(self):
        return self.split.num_rows

    def __getitem__(self, index):
        row = self.split[index]
        image = row["image"].resize((self.width, self.height), Image.BILINEAR)
        label = row["label"].resize((self.width, self.height), Image.NEAREST)
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255
        mask = torch.from_numpy(mapping[np.array(label)]).long()
        return image, mask


print("Validation dataset class defined")

### Data loaders

In [ ]:
from torch.utils.data import DataLoader

train_data = CropDataset(ds["train"], crop_size, flip=True)
eval_data = ResizeDataset(ds["validation"], val_height, val_width)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)
eval_loader = DataLoader(eval_data, batch_size=val_batch_size, shuffle=False, num_workers=num_workers)

print("Train crops per epoch: {}, batches: {}".format(len(train_data), len(train_loader)))
print("Validation photos: {}, batches: {}".format(len(eval_data), len(eval_loader)))

for name, loader in [("train", train_loader), ("validation", eval_loader)]:
    images, masks = next(iter(loader))
    print("{} batch: images {}, dtype {}, min {:.3f}, max {:.3f}".format(
        name, tuple(images.shape), images.dtype, images.min().item(), images.max().item()))
    print("{} batch: masks {}, dtype {}, values {}".format(
        name, tuple(masks.shape), masks.dtype, torch.unique(masks).tolist()))

### SegFormer-B0 with five classes

In [ ]:
import transformers
from transformers import SegformerForSemanticSegmentation

weights_name = "nvidia/mit-b0"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SegformerForSemanticSegmentation.from_pretrained(
    weights_name,
    num_labels=len(group_names),
    id2label=group_names,
    label2id={name: group_id for group_id, name in group_names.items()},
    ignore_mismatched_sizes=True,
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("transformers version: {}".format(transformers.__version__))
print("Weights: {}".format(weights_name))
print("Device: {}".format(device))
print("Parameters: {} total, {} trainable".format(total_params, trainable_params))
print("Parameters of the 3-class SegFormer-B0 printed by notebook 02: {}".format(segformer3_params))
print("Difference to the 5-class model: {}".format(total_params - segformer3_params))

### One forward pass

In [ ]:
image_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
image_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

images, masks = next(iter(train_loader))
pixel_values = (images.to(device) - image_mean) / image_std

model.eval()
with torch.no_grad():
    logits = model(pixel_values=pixel_values).logits
upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

print("Input shape: {}".format(tuple(pixel_values.shape)))
print("Logits shape: {}".format(tuple(logits.shape)))
print("Upsampled logits shape: {}".format(tuple(upsampled.shape)))
print("Logits have one channel per class: {}".format(logits.shape[1] == len(group_names)))
print("Upsampled logits have the size of the masks: {}".format(tuple(upsampled.shape[-2:]) == tuple(masks.shape[-2:])))

### Validation metrics

In [ ]:
group_count = len(group_names)


def evaluate(model, loader):
    confusion = torch.zeros(group_count, group_count, dtype=torch.int64, device=device)
    model.eval()
    with torch.no_grad():
        for images, masks in loader:
            pixel_values = (images.to(device) - image_mean) / image_std
            masks = masks.to(device)
            logits = model(pixel_values=pixel_values).logits
            upsampled = torch.nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            predicted = upsampled.argmax(dim=1)

            # Pixels with the ignore id are left out
            keep = masks != ignore_id
            pairs = masks[keep] * group_count + predicted[keep]
            confusion += torch.bincount(pairs, minlength=group_count ** 2).reshape(group_count, group_count)

    intersection = confusion.diag().float()
    union = confusion.sum(dim=0).float() + confusion.sum(dim=1).float() - intersection
    return (intersection / union).cpu().numpy()


start = time.time()
iou = evaluate(model, eval_loader)
print("Model before training")
for group_id, group_name in group_names.items():
    print("{} IoU: {:.4f}".format(group_name, iou[group_id]))
print("mIoU: {:.4f}".format(iou.mean()))
print("One validation pass took {:.1f} seconds".format(time.time() - start))

### Step timing and estimated seconds per epoch

The training steps run on a copy of the model, so the model above stays untouched. The data loading time is the time the training loop waits for the next batch, so it already includes the overlap with the model step.

In [ ]:
import copy

timing_model = copy.deepcopy(model)
timing_model.train()
timing_optimizer = torch.optim.AdamW(timing_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

data_seconds = []
step_seconds = []
batches = iter(train_loader)
for _ in range(timing_steps):
    start = time.time()
    images, masks = next(batches)
    data_seconds.append(time.time() - start)

    start = time.time()
    pixel_values = (images.to(device) - image_mean) / image_std
    loss = segmentation_loss(timing_model(pixel_values=pixel_values).logits, masks.to(device))
    timing_optimizer.zero_grad()
    loss.backward()
    timing_optimizer.step()
    if device == "cuda":
        torch.cuda.synchronize()
    step_seconds.append(time.time() - start)

start = time.time()
evaluate(timing_model, eval_loader)
validation_seconds = time.time() - start

first_step = data_seconds[0] + step_seconds[0]
steady_step = np.mean(data_seconds[1:]) + np.mean(step_seconds[1:])
steps_per_epoch = len(train_loader)
epoch_seconds = first_step + (steps_per_epoch - 1) * steady_step + validation_seconds

print("Steps timed: {}, batch size: {}".format(timing_steps, batch_size))
print("Step 1, includes start-up: data loading {:.2f} seconds, model step {:.2f} seconds".format(
    data_seconds[0], step_seconds[0]))
print("Steps 2 to {}: data loading {:.2f} seconds per step".format(timing_steps, np.mean(data_seconds[1:])))
print("Steps 2 to {}: model step {:.2f} seconds per step".format(timing_steps, np.mean(step_seconds[1:])))
print("Steps 2 to {}: total {:.2f} seconds per training step".format(timing_steps, steady_step))
print("One validation pass: {:.1f} seconds".format(validation_seconds))
if device == "cuda":
    print("Peak GPU memory during the training steps: {:.1f} GB".format(torch.cuda.max_memory_allocated() / 1024 ** 3))
print("Estimated seconds per epoch: {:.0f} ({} steps and one validation pass)".format(epoch_seconds, steps_per_epoch))
print("Estimated training time for a number of epochs on {}:".format(gpu_name))
for epochs_option in [10, 20, 30, 40, 50]:
    print("  {} epochs: about {:.0f} minutes".format(epochs_option, epochs_option * epoch_seconds / 60))

**Observation**: